# 01 - Data Analysis & Preprocessing (Phase 2)

**Project:** AI-Based Fake News Detection and News Credibility Analysis Using Machine Learning and Evidence Retrieval

**Goal of this notebook:** load the raw ISOT Fake and Real News dataset, inspect it, clean it lightly, and
split it into train/validation/test sets — using the reusable functions in `src/preprocessing/`.

This notebook does **not** perform the full exploratory data analysis (visualizations, word clouds, etc.) —
that is Phase 3. Here we only go far enough to produce clean, split data files that later phases can build on.

**Before running this notebook:** download the ISOT dataset and place `Fake.csv` and `True.csv` inside
`data/raw/`. See `README.md` -> "Phase 2 - Dataset acquisition" for exact download instructions.

## 1. Imports

In [ ]:
import sys
from pathlib import Path

import pandas as pd

# Make the project root importable so we can use the `src` package,
# regardless of which directory Jupyter was launched from.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing.load_data import load_isot_dataset, add_article_id, LABEL_FAKE, LABEL_REAL
from src.preprocessing.text_cleaning import add_clean_text_column
from src.preprocessing.quality_checks import (
    compute_data_quality_report,
    print_data_quality_report,
    remove_low_quality_rows,
)
from src.preprocessing.split_data import split_dataset, assert_no_overlap

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

pd.set_option("display.max_colwidth", 120)
print("Imports OK. Project root:", PROJECT_ROOT)

## 2. Load the raw dataset

This calls `load_isot_dataset()`, which reads `Fake.csv` and `True.csv` from `data/raw/`, validates the
required columns, adds the `label` column (1 = fake, 0 = real), and combines both files into one DataFrame.

If you see a `FileNotFoundError` below, it means the dataset files are not in `data/raw/` yet — download
them first (see README.md).

In [ ]:
try:
    df = load_isot_dataset(RAW_DATA_DIR)
    df = add_article_id(df)
    print(f"Loaded {len(df)} total records.")
except FileNotFoundError as e:
    print("Dataset not found.")
    print(e)
    raise

## 3. Dataset overview

In [ ]:
print("Shape:", df.shape)
df.head()

## 4. Column inspection

In [ ]:
df.dtypes

In [ ]:
df.info()

## 5. Missing-value analysis

In [ ]:
missing_counts = df.isnull().sum()
missing_counts

## 6. Duplicate analysis

In [ ]:
duplicate_count = df.duplicated(subset=["text"]).sum()
print(f"Duplicate articles (by exact text match): {duplicate_count}")

## 7. Class distribution

Recall the label convention: **1 = fake, 0 = real**.

In [ ]:
class_counts = df["label"].value_counts().rename({LABEL_FAKE: "fake (1)", LABEL_REAL: "real (0)"})
class_percent = (df["label"].value_counts(normalize=True) * 100).round(2).rename({LABEL_FAKE: "fake (1)", LABEL_REAL: "real (0)"})

print("Counts:")
print(class_counts)
print()
print("Percentages:")
print(class_percent)

## 8. Article length analysis (basic)

Only basic descriptive statistics here — full distribution plots belong in Phase 3 (EDA).

In [ ]:
word_counts = df["text"].fillna("").astype(str).str.split().apply(len)
char_counts = df["text"].fillna("").astype(str).str.len()

print("Word count per article:")
print(word_counts.describe())
print()
print("Character count per article:")
print(char_counts.describe())

## 9. Basic preprocessing

Steps:
1. Run a data-quality report **before** cleaning.
2. Remove rows with missing/empty text and exact duplicate articles.
3. Add a `clean_text` column (light cleaning only — the original `text` column is preserved).
4. Run the data-quality report again **after** cleaning, to see the effect.

See `src/preprocessing/text_cleaning.py` for exactly what "light cleaning" does and does not do, and why.

In [ ]:
print("BEFORE cleaning:")
report_before = compute_data_quality_report(df)
print_data_quality_report(report_before)

In [ ]:
rows_before = len(df)
df = remove_low_quality_rows(df, text_col="text")
print(f"Removed {rows_before - len(df)} rows (missing/empty text or exact duplicates).")
print(f"{len(df)} rows remain.")

In [ ]:
df = add_clean_text_column(df, source_col="text", target_col="clean_text")
df[["text", "clean_text"]].head(3)

In [ ]:
print("AFTER cleaning:")
report_after = compute_data_quality_report(df)
print_data_quality_report(report_after)

## 10. Train / validation / test split

Split: 70% train / 15% validation / 15% test, stratified on `label`, with a fixed random seed
(see `src/preprocessing/split_data.py`) for reproducibility.

We immediately check for overlap between the three splits using `article_id` (a hash of the article
text) — this is our safeguard against data leakage.

In [ ]:
train_df, val_df, test_df = split_dataset(df)
assert_no_overlap(train_df, val_df, test_df)

print("No overlap between train/validation/test splits confirmed.")
print()
print(f"Train size      : {len(train_df)} ({len(train_df) / len(df):.1%})")
print(f"Validation size : {len(val_df)} ({len(val_df) / len(df):.1%})")
print(f"Test size       : {len(test_df)} ({len(test_df) / len(df):.1%})")

In [ ]:
print("Class balance per split (share of label==1, fake):")
print(f"Train      : {train_df['label'].mean():.3f}")
print(f"Validation : {val_df['label'].mean():.3f}")
print(f"Test       : {test_df['label'].mean():.3f}")

## 11. Save processed datasets

Saved to `data/processed/`. These files are excluded from Git via `.gitignore` (they're generated,
reproducible outputs, not source material).

In [ ]:
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

train_df.to_csv(PROCESSED_DATA_DIR / "train.csv", index=False)
val_df.to_csv(PROCESSED_DATA_DIR / "validation.csv", index=False)
test_df.to_csv(PROCESSED_DATA_DIR / "test.csv", index=False)

print("Saved:")
print(f"  {PROCESSED_DATA_DIR / 'train.csv'}")
print(f"  {PROCESSED_DATA_DIR / 'validation.csv'}")
print(f"  {PROCESSED_DATA_DIR / 'test.csv'}")

## Summary

- Dataset loaded from `data/raw/Fake.csv` and `data/raw/True.csv`, combined, and labeled (1 = fake, 0 = real).
- Missing/empty text and exact duplicate articles were removed.
- A `clean_text` column was added; the original `text` column was preserved unchanged.
- Data was split 70/15/15 into train/validation/test, stratified by label, with no overlap between splits.
- Processed splits were saved to `data/processed/`.

**Next step: Phase 3 - Exploratory Data Analysis** (`notebooks/02_eda.ipynb`), which will dig into class
balance, article length distributions, common words, and source/subject analysis in more depth, with visualizations.